# 01 — Data Synthesis & Distribution Analysis

This notebook generates synthetic clinical time-series data for the AdaMed experiments.

**Source domain:** Spanish patients with continuous glucose monitors (CGM)  
**Target domain:** Ghanaian patients with sparse measurements + heuristics  

We verify that the synthetic data has:
1. Clinically plausible patterns (circadian rhythms, meal responses)
2. Meaningful distribution shift between domains
3. Appropriate missing data patterns in the target domain

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from adamed.data.synthetic_generator import ClinicalTimeSeriesGenerator
from adamed.data.heuristics import get_west_african_parameters, get_glycemic_response

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print('Imports loaded successfully.')

## 1. Generate Synthetic Data

We generate 500 source patients and 100 target proxy patients.
Each patient has 48 time steps (24h at 30-min intervals) across 5 clinical features.

In [ ]:
gen = ClinicalTimeSeriesGenerator(n_source=500, n_target_proxy=100, seed=42)
data = gen.generate_experimental_split()

source_data = data['X'][data['source_idx']]
target_data = data['X'][data['target_idx']]

print(f'Source data shape: {source_data.shape}')
print(f'Target data shape: {target_data.shape}')
print(f'Source label distribution: {np.bincount(data["y"][:len(data["source_idx"])])}')
print(f'Feature names: {data["feature_names"]}')

## 2. Visualize Sample Patient Trajectories

Comparing individual patient time-series across domains.  
Key observations:
- Source has smooth circadian patterns with regular meal spikes
- Target has higher variability and different distribution shapes

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
feature_names = data['feature_names']
t = np.linspace(0, 24, 48)

# Plot 3 source patients (top row)
for i in range(3):
    axes[0, i].set_title(f'Source Patient {i+1}', fontsize=12)
    for f in range(len(feature_names)):
        axes[0, i].plot(t, source_data[i, :, f], label=feature_names[f], alpha=0.7)
    axes[0, i].set_xlabel('Hour')
    axes[0, i].set_ylabel('Normalized Value')
    if i == 0:
        axes[0, i].legend(fontsize=7, loc='upper right')

# Plot 3 target patients (bottom row)
for i in range(3):
    axes[1, i].set_title(f'Target Patient {i+1}', fontsize=12)
    for f in range(len(feature_names)):
        axes[1, i].plot(t, target_data[i, :, f], label=feature_names[f], alpha=0.7)
    axes[1, i].set_xlabel('Hour')
    axes[1, i].set_ylabel('Normalized Value')

axes[0, 0].set_ylabel('Source Domain\nNormalized Value')
axes[1, 0].set_ylabel('Target Domain\nNormalized Value')

plt.suptitle('Sample Patient Trajectories: Source vs Target', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/sample_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Feature Distribution Comparison

Histogram overlays showing the distribution shift between domains.  
The target domain has shifted means and wider spreads — this is the gap  
that domain adaptation must bridge.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for f in range(5):
    source_vals = source_data[:, :, f].flatten()
    target_vals = target_data[:, :, f].flatten()
    target_vals = target_vals[~np.isnan(target_vals)]
    
    axes[f].hist(source_vals, bins=50, alpha=0.5, density=True, color='#2196F3', label='Source')
    axes[f].hist(target_vals, bins=50, alpha=0.5, density=True, color='#FF5722', label='Target')
    axes[f].set_title(feature_names[f], fontsize=11)
    axes[f].legend(fontsize=8)
    axes[f].grid(True, alpha=0.2)

plt.suptitle('Feature Distributions: Source vs Target Domain', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/data_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nDistribution Statistics:')
print(f'{"Feature":<20} {"Source Mean":>12} {"Target Mean":>12} {"Source Std":>12} {"Target Std":>12}')
print('-' * 70)
for f in range(5):
    s = source_data[:, :, f].flatten()
    t_vals = target_data[:, :, f].flatten()
    t_vals = t_vals[~np.isnan(t_vals)]
    print(f'{feature_names[f]:<20} {np.mean(s):>12.4f} {np.mean(t_vals):>12.4f} {np.std(s):>12.4f} {np.std(t_vals):>12.4f}')

## 4. PCA Projection of Both Domains

2D PCA shows the structural difference between domains.  
If domains cluster separately in PCA space, domain adaptation faces a real challenge.

In [ ]:
source_flat = source_data.reshape(source_data.shape[0], -1)
target_flat = target_data.reshape(target_data.shape[0], -1)
X_all = np.concatenate([source_flat, target_flat], axis=0)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_all)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X_pca[:len(source_flat), 0], X_pca[:len(source_flat), 1],
           c='#2196F3', alpha=0.4, s=15, label='Source (Spanish CGM)')
ax.scatter(X_pca[len(source_flat):, 0], X_pca[len(source_flat):, 1],
           c='#FF5722', alpha=0.4, s=15, label='Target (Ghanaian Proxy)')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('PCA Projection: Source vs Target Domain')
ax.legend(markerscale=3)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('../results/figures/pca_domains.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Explained variance: PC1={pca.explained_variance_ratio_[0]:.3f}, PC2={pca.explained_variance_ratio_[1]:.3f}')

## 5. Glycemic Response Simulation

Comparing glucose response curves for different West African staples.  
These heuristics inform the target domain distribution parameters.

In [ ]:
foods = ['kenkey', 'fufu', 'banku', 'rice_jollof']
colors = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0']

fig, ax = plt.subplots(figsize=(10, 5))
t = np.linspace(0, 4, 48)

for food, color in zip(foods, colors):
    response = get_glycemic_response(food, hours=4, samples=48)
    params = get_west_african_parameters()
    gi = params['dietary_factors'][food]['glycemic_index']
    ax.plot(t, response, label=f'{food.capitalize()} (GI={gi})', color=color, linewidth=2)

ax.set_xlabel('Hours Post-Prandial', fontsize=12)
ax.set_ylabel('Relative Glucose Response', fontsize=12)
ax.set_title('Simulated Glycemic Response: West African Staples', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 4)
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig('../results/figures/glycemic_responses.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

The synthetic data successfully captures:
1. **Distribution shift**: Target domain has shifted means and higher variability
2. **Clinical plausibility**: Circadian patterns, meal responses, and feature correlations
3. **Missing data**: Imputed for training but the original patterns are meaningful

This data will be used in the DANN experiments (notebook 02).